# NB 06: Risk Analysis & Statistical Tests

Comprehensive risk metrics, statistical significance tests, bootstrap CIs,
multiple comparison correction, and regime-conditional performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

from src.config import PROCESSED_DIR, FIGURES_DIR, RAW_DIR
from src.models.risk_metrics import (
    compute_all_metrics, sharpe_ratio, bootstrap_metric,
    max_drawdown, annualized_return,
)
from src.models.backtester import BacktestConfig, run_backtest

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')

# Load backtest results
bt_results = pd.read_parquet(PROCESSED_DIR / 'backtest_results.parquet')
wide = pd.read_parquet(PROCESSED_DIR / 'funding_rates_aligned.parquet')
print(f'Backtest results: {len(bt_results)} configurations')

## 1. Risk Metrics for Best OOS Configurations

In [ ]:
oos = bt_results[bt_results['sample_type'] == 'out_of_sample'].copy()

if not oos.empty:
    # Top 10 by Sharpe
    top10 = oos.nlargest(10, 'sharpe_ratio')
    
    metric_cols = ['exchange_a', 'exchange_b', 'asset', 'n_trades',
                   'total_return', 'annualized_return', 'annualized_volatility',
                   'sharpe_ratio', 'sortino_ratio', 'max_drawdown',
                   'calmar_ratio', 'var_95', 'cvar_95', 'hit_rate']
    metric_cols = [c for c in metric_cols if c in top10.columns]
    
    print('Top 10 OOS configurations:')
    top10[metric_cols].round(4)

## 2. Statistical Significance: t-test for Mean Return > 0

In [ ]:
# Re-run top configs to get return series for t-test
significance_results = []

if not oos.empty:
    for _, row in oos.iterrows():
        if 'ttest_p_value' in row and pd.notna(row['ttest_p_value']):
            significance_results.append({
                'exchange_a': row.get('exchange_a'),
                'exchange_b': row.get('exchange_b'),
                'asset': row.get('asset'),
                't_stat': row.get('ttest_t_stat'),
                'p_value': row.get('ttest_p_value'),
                'significant_5pct': row.get('ttest_p_value', 1) < 0.05,
                'significant_1pct': row.get('ttest_p_value', 1) < 0.01,
                'sharpe_ratio': row.get('sharpe_ratio'),
            })

sig_df = pd.DataFrame(significance_results)
if not sig_df.empty:
    print(f'Significant at 5%: {sig_df["significant_5pct"].sum()}/{len(sig_df)}')
    print(f'Significant at 1%: {sig_df["significant_1pct"].sum()}/{len(sig_df)}')

## 3. Multiple Comparison Correction (Bonferroni / BH)

In [ ]:
if not sig_df.empty and len(sig_df) > 1:
    p_values = sig_df['p_value'].values
    
    # Bonferroni
    bonf_reject, bonf_pvals, _, _ = multipletests(p_values, method='bonferroni', alpha=0.05)
    sig_df['bonferroni_reject'] = bonf_reject
    sig_df['bonferroni_p'] = bonf_pvals
    
    # Benjamini-Hochberg
    bh_reject, bh_pvals, _, _ = multipletests(p_values, method='fdr_bh', alpha=0.05)
    sig_df['bh_reject'] = bh_reject
    sig_df['bh_p'] = bh_pvals
    
    print(f'Significant after Bonferroni: {bonf_reject.sum()}/{len(sig_df)}')
    print(f'Significant after BH FDR: {bh_reject.sum()}/{len(sig_df)}')
    
    sig_df.sort_values('p_value')[[
        'exchange_a', 'exchange_b', 'asset', 'sharpe_ratio',
        'p_value', 'bonferroni_p', 'bh_p', 'bonferroni_reject', 'bh_reject'
    ]].round(4)

## 4. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap CIs for top 5 OOS configs (if available in results)
if not oos.empty:
    top5 = oos.nlargest(5, 'sharpe_ratio')
    
    boot_records = []
    for _, row in top5.iterrows():
        # Check if bootstrap results are stored
        sr_boot = row.get('sharpe_bootstrap')
        tr_boot = row.get('total_return_bootstrap')
        
        record = {
            'pair': f"{row.get('exchange_a')}/{row.get('exchange_b')} {row.get('asset')}",
        }
        
        if isinstance(sr_boot, dict):
            record['sharpe_point'] = sr_boot.get('point_estimate')
            record['sharpe_ci_lower'] = sr_boot.get('ci_lower')
            record['sharpe_ci_upper'] = sr_boot.get('ci_upper')
        else:
            record['sharpe_point'] = row.get('sharpe_ratio')
        
        if isinstance(tr_boot, dict):
            record['return_point'] = tr_boot.get('point_estimate')
            record['return_ci_lower'] = tr_boot.get('ci_lower')
            record['return_ci_upper'] = tr_boot.get('ci_upper')
        else:
            record['return_point'] = row.get('total_return')
        
        boot_records.append(record)
    
    boot_df = pd.DataFrame(boot_records)
    print('Bootstrap 95% CIs for Top 5 OOS:')
    boot_df.round(4)

## 5. Correlation with BTC Buy-and-Hold

In [ ]:
# Load BTC price for comparison
price_files = list((RAW_DIR / 'prices').rglob('BTC.parquet'))

if price_files and not oos.empty:
    btc_prices = pd.read_parquet(price_files[0])
    btc_prices['timestamp'] = pd.to_datetime(btc_prices['timestamp'], utc=True)
    btc_prices = btc_prices.set_index('timestamp').sort_index()
    btc_8h = btc_prices['close'].resample('8h').last().pct_change().dropna()
    
    # Re-run best OOS config to get return series
    best = oos.iloc[oos['sharpe_ratio'].idxmax()] if not oos.empty else None
    if best is not None:
        config = BacktestConfig(
            entry_threshold=best.get('config_entry_threshold', 0.0005),
            leverage=best.get('config_leverage', 1),
        )
        result = run_backtest(
            wide, best['exchange_a'], best['exchange_b'], best['asset'],
            config=config, sample_type='full',
        )
        
        if result.returns is not None:
            # Align
            common = result.returns.index.intersection(btc_8h.index)
            if len(common) > 50:
                corr = result.returns.loc[common].corr(btc_8h.loc[common])
                print(f'Correlation of strategy returns with BTC: {corr:.4f}')
                print(f'Expected: ~0 (market-neutral)')
                
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.scatter(btc_8h.loc[common] * 100, result.returns.loc[common] * 100,
                          alpha=0.3, s=5)
                ax.set_xlabel('BTC 8h Return (%)')
                ax.set_ylabel('Strategy 8h Return (%)')
                ax.set_title(f'Strategy vs BTC Returns (corr = {corr:.3f})')
                ax.axhline(0, color='k', alpha=0.3, linestyle='--')
                ax.axvline(0, color='k', alpha=0.3, linestyle='--')
                plt.tight_layout()
                plt.savefig(FIGURES_DIR / 'strategy_vs_btc.pdf', bbox_inches='tight')
                plt.show()
else:
    print('No BTC price data or backtest results available for comparison')

## 6. Regime-Conditional Performance

In [ ]:
if price_files and not oos.empty:
    # Classify regimes by BTC volatility
    btc_vol = btc_8h.rolling(30).std()  # 30-period rolling vol
    vol_median = btc_vol.median()
    
    high_vol = btc_vol > vol_median
    
    if result.returns is not None:
        common = result.returns.index.intersection(high_vol.index)
        if len(common) > 50:
            strat_ret = result.returns.loc[common]
            hv_mask = high_vol.loc[common]
            
            hv_returns = strat_ret[hv_mask]
            lv_returns = strat_ret[~hv_mask]
            
            print('Regime-Conditional Performance:')
            print(f'  High-vol periods: {len(hv_returns)}, mean={hv_returns.mean():.6f}, Sharpe={sharpe_ratio(hv_returns):.2f}')
            print(f'  Low-vol periods:  {len(lv_returns)}, mean={lv_returns.mean():.6f}, Sharpe={sharpe_ratio(lv_returns):.2f}')
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            
            (1 + hv_returns).cumprod().plot(ax=ax1, label='High Vol')
            (1 + lv_returns).cumprod().plot(ax=ax1, label='Low Vol')
            ax1.set_title('Cumulative Returns by Volatility Regime')
            ax1.legend()
            
            ax2.hist(hv_returns * 100, bins=50, alpha=0.5, label='High Vol', density=True)
            ax2.hist(lv_returns * 100, bins=50, alpha=0.5, label='Low Vol', density=True)
            ax2.set_title('Return Distribution by Regime')
            ax2.set_xlabel('Return (%)')
            ax2.legend()
            
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / 'regime_performance.pdf', bbox_inches='tight')
            plt.show()